# CNN Implementation

### 卷积层输出尺寸公式

$$
H_{out} = \left\lfloor \frac{H_{in} + 2 \cdot padding - kernel\_size}{stride} + 1 \right\rfloor
$$

$$
W_{out} = \left\lfloor \frac{W_{in} + 2 \cdot padding - kernel\_size}{stride} + 1 \right\rfloor
$$

where:
- $H_{in}$ and $W_{in}$ are the height and width of the input tensor
- $H_{out}$ and $W_{out}$ are the height and width of the output tensor
- $kernel\_size$ is the size of the convolutional kernel
- $stride$ is the stride of the convolution
- $padding$ is the padding added to the input tensor

In [ ]:
import torch 
from torch import nn
import torch.nn.functional as F

# cnn from scratch
class CNN(nn.Module):
    def __init__(self, in_channels, embed_dim, kernel_size, stride):
        super().__init__()
        self.kernel = nn.Parameter(torch.randn(embed_dim, in_channels, kernel_size, kernel_size))
        self.in_channels = in_channels
        self.out_channels = embed_dim
        self.k = kernel_size
        self.s = stride
    def forward(self,x):
        # x: (N, C, H, W)
        n, c, h, w = x.size()
        patches = F.unfold(x, kernel_size=self.k, stride=self.s)  # (N, C*k*k, out_h*out_w)
        kernel_flat = self.kernel.view(self.out_channels, -1)  # (out_channels, C*k*k)
        out = kernel_flat @ patches  # (N, out_channels, out_h*out_w)
        out_h = (h - self.k) // self.s + 1
        out_w = (w - self.k) // self.s + 1
        # out: (N, out_channels, out_h, out_w)
        out = out.view(n, self.out_channels, out_h, out_w)

        return out

# test
x = torch.randn(2, 3, 10, 10)
cnn = CNN(3, 12, 3, 1)
total_params = sum(p.numel() for p in cnn.parameters())
print(f'Total parameters: {total_params}') # 3 * 3 * 3 * 12 = 324
output = cnn(x)
print(output.size())